In [10]:
import pandas as pd

import numpy as np

import re

import pickle

import faiss

from sentence_transformers import SentenceTransformer

from transformers import pipeline

from sklearn.metrics.pairwise import cosine_similarity

In [11]:
df = pd.read_csv(
    "../processed/final_dataset.csv"
)

print(df.shape)

print(df.head())

(391612, 3)
                                                text  label    source
0  Which magazine was started first Arthur's Maga...      0  HaluEval
1  Which magazine was started first Arthur's Maga...      1  HaluEval
2  The Oberoi family is part of a hotel company t...      0  HaluEval
3  The Oberoi family is part of a hotel company t...      1  HaluEval
4  Musician and satirist Allie Goertz wrote a son...      0  HaluEval


In [12]:
subset_df = df.sample(
    50000,
    random_state=42
)

print(subset_df.shape)

(50000, 3)


In [13]:
index = faiss.read_index(
    "../faiss_index/wiki.index"
)

print("FAISS index loaded!")

FAISS index loaded!


In [14]:
with open(
    "../embeddings/wiki_chunks.pkl",
    "rb"
) as f:

    wiki_chunks = pickle.load(f)

print("Total chunks:", len(wiki_chunks))

Total chunks: 1076104


In [15]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

print("Embedding model loaded!")

c:\Users\SujanRam\OneDrive\Documents\Hallucination project\.venv\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding model loaded!


In [16]:
nli_pipeline = pipeline(
    "text-classification",

    model="MoritzLaurer/deberta-v3-base-mnli-fever-anli",

    device=0
)

print("NLI model loaded!")

NLI model loaded!


In [17]:
def retrieve_evidence(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        device="cpu"
    )

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        50
    )

    query_words = set(
        re.findall(r'\w+', query.lower())
    )

    scored_passages = []

    for rank, idx in enumerate(indices[0]):

        passage = wiki_chunks[int(idx)]

        passage_lower = passage.lower()

        passage_words = set(
            re.findall(r'\w+', passage_lower)
        )

        # Use FAISS similarity
        sim = 1 / (
            1 + distances[0][rank]
        )

        # Keyword overlap
        overlap = len(
            query_words.intersection(
                passage_words
            )
        )

        # Entity bonus
        entity_bonus = sum(
            1
            for word in query_words
            if word in passage_lower
        )

        # Capital keyword bonus
        capital_bonus = (
            3
            if "capital" in passage_lower
            else 0
        )

        # Final score
        final_score = (
            0.5 * sim
            +
            0.3 * overlap
            +
            0.15 * entity_bonus
            +
            0.05 * capital_bonus
        )

        scored_passages.append(
            (final_score, passage)
        )

    scored_passages = sorted(
        scored_passages,
        key=lambda x: x[0],
        reverse=True
    )

    top_passages = [
        p[1]
        for p in scored_passages[:top_k]
    ]

    return top_passages

In [18]:
def extract_features(claim):

    passages = retrieve_evidence(
        claim,
        top_k=2
    )

    claim_embedding = embedding_model.encode(
        [claim],
        convert_to_numpy=True,
        device="cpu"
    )

    similarities = []

    entailment_probs = []

    contradiction_probs = []

    neutral_probs = []

    for passage in passages:

        passage_embedding = embedding_model.encode(
            [passage],
            convert_to_numpy=True,
            device="cpu"
        )

        sim = cosine_similarity(
            claim_embedding,
            passage_embedding
        )[0][0]

        similarities.append(sim)

        # NLI verification
        result = nli_pipeline(
            {
                "text": claim,
                "text_pair": passage
            }
        )

        label = result["label"].lower()

        score = result["score"]

        entailment = 0
        contradiction = 0
        neutral = 0

        if "entail" in label:

            entailment = score

        elif "contrad" in label:

            contradiction = score

        else:

            neutral = score

        entailment_probs.append(
            entailment
        )

        contradiction_probs.append(
            contradiction
        )

        neutral_probs.append(
            neutral
        )

    # Core features

    max_similarity = max(similarities)

    mean_similarity = np.mean(similarities)

    max_entailment = max(entailment_probs)

    max_contradiction = max(contradiction_probs)

    max_neutral = max(neutral_probs)

    claim_length = len(
        claim.split()
    )

    entity_count = len(
        re.findall(
            r'\b[A-Z][a-z]+\b',
            claim
        )
    )

    number_count = len(
        re.findall(
            r'\d+',
            claim
        )
    )

    negation_present = int(
        any(
            word in claim.lower()
            for word in [
                "not",
                "no",
                "never",
                "none"
            ]
        )
    )

    # Additional features

    evidence_count = len(passages)

    avg_word_length = np.mean(
        [
            len(word)
            for word in claim.split()
        ]
    )

    punctuation_count = len(
        re.findall(
            r'[^\w\s]',
            claim
        )
    )

    unique_word_ratio = len(
        set(claim.split())
    ) / max(len(claim.split()), 1)

    uppercase_ratio = sum(
        1
        for c in claim
        if c.isupper()
    ) / max(len(claim), 1)

    temporal_expression_present = int(
        bool(
            re.search(
                r'\b(19|20)\d{2}\b',
                claim
            )
        )
    )

    evidence_conflict_score = (
        max_contradiction
    )

    source_reliability_score = (
        max_similarity
    )

    claim_specificity = (
        unique_word_ratio
    )

    return {

        "max_similarity":
            max_similarity,

        "mean_similarity":
            mean_similarity,

        "max_entailment":
            max_entailment,

        "max_contradiction":
            max_contradiction,

        "max_neutral":
            max_neutral,

        "claim_length":
            claim_length,

        "entity_count":
            entity_count,

        "number_count":
            number_count,

        "negation_present":
            negation_present,

        "evidence_count":
            evidence_count,

        "avg_word_length":
            avg_word_length,

        "punctuation_count":
            punctuation_count,

        "unique_word_ratio":
            unique_word_ratio,

        "uppercase_ratio":
            uppercase_ratio,

        "temporal_expression_present":
            temporal_expression_present,

        "evidence_conflict_score":
            evidence_conflict_score,

        "source_reliability_score":
            source_reliability_score,

        "claim_specificity":
            claim_specificity
    }

In [19]:
sample_claim = subset_df.iloc[0]["text"]

features = extract_features(
    sample_claim
)

print(features)

{'max_similarity': 0.48317587, 'mean_similarity': 0.45257503, 'max_entailment': 0.9467876553535461, 'max_contradiction': 0, 'max_neutral': 0, 'claim_length': 24, 'entity_count': 2, 'number_count': 0, 'negation_present': 0, 'evidence_count': 2, 'avg_word_length': 5.5, 'punctuation_count': 4, 'unique_word_ratio': 0.6666666666666666, 'uppercase_ratio': 0.012903225806451613, 'temporal_expression_present': 0, 'evidence_conflict_score': 0, 'source_reliability_score': 0.48317587, 'claim_specificity': 0.6666666666666666}


In [20]:
feature_rows = []

In [ ]:
feature_rows

[{'max_similarity': 0.49377784,
  'mean_similarity': 0.49040627,
  'max_entailment': 0,
  'max_contradiction': 0,
  'max_neutral': 0.9990593791007996,
  'claim_length': 13,
  'entity_count': 7,
  'number_count': 0,
  'negation_present': 0,
  'evidence_count': 2,
  'avg_word_length': 5.846153846153846,
  'punctuation_count': 3,
  'unique_word_ratio': 0.8461538461538461,
  'uppercase_ratio': 0.07954545454545454,
  'temporal_expression_present': 0,
  'evidence_conflict_score': 0,
  'source_reliability_score': 0.49377784,
  'claim_specificity': 0.8461538461538461,
  'label': 0},
 {'max_similarity': 0.5421347,
  'mean_similarity': 0.5100715,
  'max_entailment': 0,
  'max_contradiction': 0,
  'max_neutral': 0.9989051818847656,
  'claim_length': 17,
  'entity_count': 7,
  'number_count': 0,
  'negation_present': 0,
  'evidence_count': 2,
  'avg_word_length': 5.235294117647059,
  'punctuation_count': 3,
  'unique_word_ratio': 0.7647058823529411,
  'uppercase_ratio': 0.06666666666666667,
  'tem

In [21]:
for i, row in subset_df.iterrows():

    claim = row["text"]

    label = row["label"]

    try:

        features = extract_features(
            claim
        )

        features["label"] = label

        feature_rows.append(
            features
        )

        # Save checkpoint every 500 samples
        if len(feature_rows) % 500 == 0:

            temp_df = pd.DataFrame(
                feature_rows
            )

            temp_df.to_csv(
                "../processed/features_checkpoint.csv",
                index=False
            )

            print(
                f"Checkpoint saved at {len(feature_rows)} samples"
            )

        if i % 50 == 0:

            print(
                f"Processed {i}"
            )

    except Exception as e:

        print("Error:", e)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processed 385600
Processed 281500
Processed 315150


KeyboardInterrupt: 

In [ ]:
print(features_df.head())

   max_similarity  mean_similarity  max_entailment  max_contradiction  \
0        0.493778         0.490406             0.0           0.000000   
1        0.542135         0.510072             0.0           0.000000   
2        0.416591         0.401467             0.0           0.000000   
3        0.386189         0.385280             0.0           0.502376   
4        0.400931         0.397153             0.0           0.000000   

   max_neutral  claim_length  entity_count  number_count  negation_present  \
0     0.999059            13             7             0                 0   
1     0.998905            17             7             0                 0   
2     0.994209            18             3             0                 0   
3     0.928322            26             5             0                 0   
4     0.988057            23            11             0                 0   

   evidence_count  avg_word_length  punctuation_count  unique_word_ratio  \
0               

In [ ]:
features_df.to_csv(
    "../processed/features_dataset.csv",
    index=False
)

In [ ]:
print("Feature engineering completed successfully!")

Feature engineering completed successfully!
